# 0. Import and Reload

In [1]:
import os
import sys
import json
import pickle
import random
from tqdm import tqdm
from copy import deepcopy
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import random
import pandas as pd
import matplotlib.pyplot as plt
import importlib
import scipy
from sklearn.metrics import f1_score

import utils.oracle as oracle
import llm_connector.Collector as collector

In [2]:
importlib.reload(oracle)
importlib.reload(collector)

<module 'llm_connector.Collector' from "/Users/sharkiefff/Desktop/NUS/Lab_Dissertation/KDD'25/LLM-Query-Reduction-Baseline/src/llm_connector/Collector.py">

In [3]:
# load dataset
mba_df = pd.read_csv('../data/MBA/data.csv', sep=';')

# clean nan rows
if mba_df.isna().sum().sum() > 0:
    print('all nan eliminated')
    mba_df = mba_df.dropna()

# transfer types
mba_df['BillNo'] = mba_df['BillNo'].astype('int32')
mba_df['Itemname'] = mba_df['Itemname'].astype('string')
mba_df['Quantity'] = mba_df['Quantity'].astype('int32')
mba_df['Date'] = mba_df['Date'].astype('string')
mba_df['Price'] = mba_df['Price'].astype('string')
mba_df['CustomerID'] = mba_df['CustomerID'].astype('int32')

# unique
item_names_set = mba_df['Itemname'].unique()
uid_set = mba_df['CustomerID'].unique()
trans_id_set = mba_df['BillNo'].unique()
len(mba_df), len(trans_id_set), len(item_names_set), len(uid_set)

# for user nodes
user_ids = mba_df['CustomerID'].unique()
user_num = len(user_ids)
print(f'totally {user_num} unique users')
user_ids.sort()
user_ids_kv = {}
for ui in range(user_num):
    user_ids_kv[user_ids[ui]] = ui

# for transaction nodes
trans_ids = mba_df['BillNo'].unique()
trans_num = len(trans_ids)
print(f'totally {trans_num} unique transactions')
trans_ids.sort()
trans_kv = {}
for ti in range(trans_num):
    trans_kv[trans_ids[ti]] = ti

# for item nodes
item_names = mba_df['Itemname'].unique()
item_num = len(item_names)
print(f'totally {item_num} unique items')
# item_names.sort()
items_kv = {}
for ii in range(item_num):
    items_kv[item_names[ii]] = ii

/var/folders/lh/1l1yxhh50vdcdltw404st9z80000gn/T/ipykernel_63505/3082873686.py:2: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  mba_df = pd.read_csv('../data/MBA/data.csv', sep=';')


all nan eliminated
totally 4297 unique users
totally 18163 unique transactions
totally 3846 unique items


In [4]:
# MODEL_NAME = 'gpt-4o-mini'
# MODEL_NAME = 'gpt-4o'
# MODEL_NAME = 'llama-3-8B'
MODEL_NAME = 'llama-3.1-70B'
# MODEL_NAME = 'mixtral-8x7B'

In [5]:
# Define the API keys and corresponding model names
api_keys = {
    'deep_infra': 'BoH8g6vLaKaI2l1Cgd6v000ES3MF9S8l'
}

model_names = {
    # 'gpt-4o-mini': 'openai',
    # 'gpt-4o': 'openai',
    'llama-3.1-70B': 'deep_infra',
    # 'llama-3-8B': 'deep_infra',
    # 'mixtral-8x7B': 'deep_infra'
}

In [6]:
def create_oracle(model_name, api_key):
    return oracle.Oracle(model_name, api_key)

def get_api_key(model_name):
    provider = model_names[model_name]
    return api_keys[provider]

In [7]:
# Example usage
model_name = MODEL_NAME
api_key = get_api_key(model_name)
oracle_instance = create_oracle(model_name, api_key)
print(f"Created oracle instance for model: {model_name}")

Created oracle instance for model: llama-3.1-70B


# 1. Prompt

In [8]:
# Group by CustomerID, BillNo, Itemname and sum the quantity
grouped_item_df = mba_df.groupby(['CustomerID', 'Itemname']).agg({'Quantity': 'sum'}).reset_index()

# print the customer id = 13261
print(grouped_item_df[grouped_item_df['CustomerID'] == 13261])

       CustomerID                      Itemname  Quantity
41387       13261         DOORMAT 3 SMILEY CATS        20
41388       13261                DOORMAT HEARTS        10
41389       13261               DOORMAT TOPIARY        20
41390       13261       DOORMAT WELCOME PUPPIES        20
41391       13261  HOMEMADE JAM SCENTED CANDLES        48
41392       13261        PAPER CHAIN KIT EMPIRE        40
41393       13261                 PARTY BUNTING        40
41394       13261    VINTAGE UNION JACK BUNTING        12


In [9]:
def describe_user(uid, df):
    user_df = df[df['CustomerID'] == uid]
    items = dict(user_df[['Itemname', 'Quantity']].values)
    # add description like "36 PENCILS TUBE SKULLS purchased 16 times"
    # items_description = '; '.join([f'{item} purchased {count} times' for item, count in items.items()])
    items_description = '; '.join([f'{item}, {count} times' for item, count in items.items()])
    return f'The user {uid} has purchased {len(items)} items: each item name followed by its purchased times: he bought ' + items_description

def describe_users(uids, df):
    descriptions = []
    for uid in uids:
        descriptions.append(describe_user(uid, df))
    return descriptions

# unit test
print(len(describe_users(user_ids, grouped_item_df)))

4297


In [10]:
describe_users([user_ids[11], user_ids[666]], grouped_item_df)

["The user 12360 has purchased 105 items: each item name followed by its purchased times: he bought 36 PENCILS TUBE SKULLS, 16 times; ALPHABET HEARTS STICKER SHEET, 12 times; ASSORTED COLOUR MINI CASES, 4 times; BAKING SET 9 PIECE RETROSPOT, 3 times; BAKING SET SPACEBOY DESIGN, 3 times; BALLOON ART MAKE YOUR OWN FLOWERS, 10 times; BIRTHDAY CARD, RETRO SPOT, 12 times; BLUE POLKADOT BOWL, 8 times; BLUE POLKADOT CUP, 16 times; BLUE POLKADOT PLATE, 16 times; BLUE VINTAGE SPOT BEAKER, 8 times; BOX OF 6 CHRISTMAS CAKE DECORATIONS, 4 times; BOX OF 6 MINI VINTAGE CRACKERS, 6 times; BUTTERFLIES STICKERS, 12 times; CARD CHRISTMAS VILLAGE, 12 times; CARD MOTORBIKE SANTA, 12 times; CARD PARTY GAMES, 12 times; CHILDRENS APRON APPLES DESIGN, 8 times; CHILDRENS APRON SPACEBOY DESIGN, 8 times; CHILDRENS CUTLERY CIRCUS PARADE, 4 times; CHILDRENS CUTLERY POLKADOT BLUE, 4 times; CHILDRENS CUTLERY POLKADOT GREEN, 12 times; CHILDRENS CUTLERY POLKADOT PINK, 20 times; CHILDRENS CUTLERY RETROSPOT RED, 4 times

In [11]:
prompt_str_1 = """
You are an assistant skilled at summarizing, capable of deducing high-level consumer keywords based on a user's purchases. 
"""

query_str_1 = """
Take a deep breath and work according to the instructions step by step.
Now you will conduct a series of analyses on the Market Basket Analysis (MBA) dataset. This dataset contains data from a retailer, where each user's purchasing transactions and the bought items are recorded. 
From this dataset, I will provide you the purchase information from about 100(2.5 percent) users, per-user's purchasing data has been grouped by their ID and transferred to a natural language description for your better understanding of their purchasing behaviors.
Your task is to generate representative and accurate 20 user personas according to these users' purchasing patterns. Please notice that we give you 2 important targets you should consider and optimize:
- 'High Coverage': We hope that your generated persona set can cover as many users as possible. We define the 'coverage' as the total number of the users which can be labeled with at least one of your generated persona set.
- 'High Accuracy': We hope that each of your generated persona has a precise definition. An ambiguous or sweeping persona definition should be avoided.
Repeat your task one more time, is to generate a proper set of 20 representative and accurate user personas existing in the data subset and explain them quantitatively. For each persona, you should write a corresponding definition and count the cumulative number of occurrences of this persona among users.
You can choose your own written style to write the definition. And you should only output the persona set without extra explanation. The personas should be returned in the decreasing order of their occurence times, the persona with highest occurrence list first.
An output example:
1. Home Comforts Enthusiast - Occurrence: 7
   - Buys items focused on creating a cozy and inviting home atmosphere, such as wicker hearts, chalkboards, vintage decorative pieces, and heart-shaped ornaments.
2. Craft and DIY Hobbyist - Occurrence: 6
   - Often purchases crafting materials, DIY kits, sewing items, plush toys, and bespoke stationery sets for personal projects or to entertain children.
3. Kitchenware Collector - Occurrence: 4
   - Has a preference for vintage and retro kitchenware, including cake stands, tea plates, mug sets, kitchen scales, and ceramic storage containers.
...
(20 personas in total)

We define the 'occurrence times' of a persona as the cumulative NUMBER OF USERS who belong to this persona, INSTEAD OF THE NUMBER OF ALL RELATED PURCHASED ITEMS!!!!
Now considering the user purchasing data given below:
"""

In [12]:
prompt_str_2 = """
You are an assistant skilled at reading, observing and summarizing, capable of finding similar or repeated description of user personas, and good at finding the most representative ones.
"""

query_str_2 = """
Take a deep breath and work according to the instructions step by step.
Now we have 40 persona_sets, each containing 20 personas, and I will randomly select five persona_sets, each containing 20 personas, for a total of 100 personas. Your task is to select the 20 most representative personas from these 100 personas and output the results.
If you find that the content of a certain group is not 20 personas but less than 20, or even irrelevant information, you should ignore this group of information and only refer to personas in other groups.
Note that you may find that the personas you read have some similarities or even some duplications. You need to find these similar or duplicate personas and select the 20 most representative personas accordingly.
You should not refer to any information related to the number of occurrences in these personas, as this information is very likely to be unreasonable. You should ignore the occurrence times information and select the 20 most representative personas based only on their descriptions.
Here are the five sets of results that make up the 100 personas you need to choose from:
"""

In [13]:
prompt_str_3 = """
You are an assistant skilled at reading, observing and summarizing, capable of finding similar or repeated description of user personas, and good at finding the most representative ones.
"""

query_str_3 = """
Take a deep breath and work according to the instructions step by step.
Now we have 8 persona sets, each containing 20 personas, for a total of 160 personas. Your task is to select the 20 most representative personas from these 160 personas and output the results.
Note that you may find that the personas you read have some similarities or even some duplications. You need to find these similar or duplicate personas and select the 20 most representative personas accordingly, that is to say, these 20 personas occurs most times and can cover most of them.
Here are the eight sets of personas that make up the 160 personas you need to choose from:
"""


# 2. Test

In [ ]:
item_query_demo = "Tell me today's weather in San Francisco."

In [ ]:
import tiktoken
encoding = tiktoken.get_encoding("cl100k_base")
tokens = encoding.encode(item_query_demo)
token_count = len(tokens)
print(f"Token number: {token_count}")

Token number: 9


In [ ]:
oracle_instance.query('you are an assisant', item_query_demo, temp=1.0, top_p=0.9)

{'query': 'WHITE HANGING HEART T-LIGHT HOLDER',
 'answer': "A lovely home decor item!\n\nA White Hanging Heart T-Light Holder is a beautiful and romantic decorative piece that can add a warm and cozy ambiance to any room. Here are some key features and ideas for using this item:\n\n**Description:**\n\n* A delicate, white heart-shaped holder designed to hold a tea light candle\n* Typically made of ceramic, porcelain, or glass\n* Features a hanging loop or chain, allowing it to be suspended from a hook or a nail\n\n**Ideas for use:**\n\n1. **Romantic decor:** Hang the heart-shaped holder in a bedroom or living room to create a warm and intimate atmosphere.\n2. **Wedding decor:** Use multiple holders as a beautiful and unique centerpiece for wedding tables or as a decorative element in a wedding arch.\n3. **Outdoor decor:** Hang the holder from a tree branch or a patio hook to add a warm glow to outdoor spaces.\n4. **Seasonal decor:** Use the holder as a decorative element for Valentine's

# 3. Initial persona set generation

### 3.1 step 1: 2.5% * 40

In [17]:
random.seed(42)
sample_size = int(len(user_ids) * 0.025)
print(f'Sample size: {sample_size}')

sample_times = 40

user_ids_copy = list(user_ids.copy())

all_samples = []

for i in range(sample_times):
    sampled_uids = random.sample(user_ids_copy, sample_size)
    user_ids_copy = [uid for uid in user_ids_copy if uid not in sampled_uids]
    all_samples.append(sampled_uids)

Sample size: 107


In [18]:
# check if the sample is correct, no duplicate and no missing
sample_set = set()
for sample in all_samples:
    sample_set.update(sample)
print(len(sample_set), len(user_ids))

4280 4297


In [19]:
# 利用 describe_users 函数生成 query_list
query_list = []
for sample in all_samples:
    descriptions = describe_users(sample, grouped_item_df)
    combined_description = ' '.join(descriptions)
    query_list.append(combined_description)

# 打印 query_list 的长度
print(len(query_list))

# 打印每个 query 的长度
for i, query in enumerate(query_list):
    print(f'Query {i+1} length: {len(query)}')

40
Query 1 length: 267881
Query 2 length: 229434
Query 3 length: 215166
Query 4 length: 221828
Query 5 length: 277515
Query 6 length: 259871
Query 7 length: 243429
Query 8 length: 279267
Query 9 length: 221438
Query 10 length: 277840
Query 11 length: 281161
Query 12 length: 274236
Query 13 length: 264262
Query 14 length: 245945
Query 15 length: 262901
Query 16 length: 247300
Query 17 length: 242601
Query 18 length: 229140
Query 19 length: 292741
Query 20 length: 275649
Query 21 length: 262249
Query 22 length: 289497
Query 23 length: 288003
Query 24 length: 194677
Query 25 length: 286707
Query 26 length: 260878
Query 27 length: 297200
Query 28 length: 270587
Query 29 length: 227106
Query 30 length: 235612
Query 31 length: 247403
Query 32 length: 301776
Query 33 length: 267500
Query 34 length: 239723
Query 35 length: 252214
Query 36 length: 302446
Query 37 length: 213902
Query 38 length: 263053
Query 39 length: 269582
Query 40 length: 302724


In [20]:
# for each list in the query_list, call oracle_instance.query_all, and store them named as ../data/MBA/1204_{MODEL_NAME}_item_results_i.pkl, where i is the index of the list
results = oracle_instance.query_all(prompt_str_1 + query_str_1, query_list, temp=1.0, top_p=0.1)
with open(f'../data/MBA/1204/{MODEL_NAME}_item_results_2.pkl', 'wb') as f:
    pickle.dump(results, f)

Total queries: 40, start collecting...


Processing Items: 100%|██████████| 40/40 [03:15<00:00,  4.90s/it]


### 3.2. Sample every 5 files and join them as one

In [43]:
# read ../data/MBA/1204_1/llama-3.1-70B_item_results_0.pkl
# select every 5 answers and combine them into a list, then you should get 8 lists (40/5)

# read all the results
results_list = []
with open(f'../data/MBA/1204/{MODEL_NAME}_item_results_2.pkl', 'rb') as f:
    results = pickle.load(f)
    results_list.append(results)

# seperate the results into 8 lists
seperated_results = []
for results in results_list:
    answers = [entry['answer'] for entry in results]
    for i in range(0, len(answers), 5):
        combined_answers_str = ' '.join(answers[i:i+5])
        seperated_results.append(combined_answers_str)

print(len(seperated_results))

8


In [44]:
results_2 = oracle_instance.query_all(prompt_str_2 + query_str_2, seperated_results, temp=1.0, top_p=0.1)

Total queries: 8, start collecting...


Processing Items: 100%|██████████| 8/8 [00:31<00:00,  3.95s/it]


### 3.3. Last step, summarize 20 persona from generated 160 personas.

In [45]:
# concat all the answer from results_2, concat them as a single string
all_answers = ''
for result in results_2:
    all_answers += result['answer']

In [46]:
final_result = oracle_instance.query(prompt_str_3 + query_str_3, all_answers, temp=1.0, top_p=0.1)

In [47]:
print(final_result['answer'])

After analyzing the provided data, I have identified the 20 most representative personas. Here they are:

1. **Home Comforts Enthusiast**: Buys items focused on creating a cozy and inviting home atmosphere.
2. **Craft and DIY Hobbyist**: Often purchases crafting materials, DIY kits, sewing items, plush toys, and bespoke stationery sets.
3. **Kitchenware Collector**: Has a preference for vintage and retro kitchenware.
4. **Gardening Enthusiast**: Purchases gardening-related items, including planters, gardening gloves, and gardening tools.
5. **Home Decor Enthusiast**: Buys items to decorate their home, including wall art, vases, and decorative accents.
6. **Foodie**: Purchases gourmet food items, cookbooks, and kitchen gadgets.
7. **Traveler**: Buys travel-related items, including luggage tags, travel wallets, and travel accessories.
8. **Bookworm**: Purchases books, book-related accessories, and book-themed gifts.
9. **Pet Lover**: Buys pet-related items, including pet accessories, pet